In [1]:
import tensorflow as tf
import rasterio
import numpy as np

def load_sar_tiff_tf(path):
    def _read(p):
        with rasterio.open(p.decode()) as src:
            vv = src.read(1).astype(np.float32)
            vh = src.read(2).astype(np.float32)

        # Fixed-range normalization
        vv = np.clip(vv, -35.0, 5.0)
        vh = np.clip(vh, -40.0, 0.0)

        vv = (vv + 35.0) / 40.0
        vh = (vh + 40.0) / 40.0

        img = np.stack([vv, vh], axis=-1)

        # Resize 2048 → 512
        img = tf.image.resize(img, (512, 512), method="bilinear").numpy()

        return img

    img = tf.numpy_function(_read, [path], tf.float32)
    img.set_shape([512, 512, 2])
    return img

In [2]:
import os

IMG_SIZE = (512, 512)
BATCH_SIZE = 8
EPOCHS = 50

DATASET_ROOT = "/Volumes/Windows8_OS/Dataset/Dataset-OG"
TRAIN_DIR = os.path.join(DATASET_ROOT, "Train", "Images")
TEST_DIR  = os.path.join(DATASET_ROOT, "Test", "Images")

In [3]:
def augment(image, label):
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_flip_up_down(image)
    image = tf.image.rot90(image, tf.random.uniform([], 0, 4, tf.int32))

    # Mild speckle-like noise
    noise = tf.random.normal(tf.shape(image), stddev=0.02)
    image = tf.clip_by_value(image + noise, 0.0, 1.0)

    return image, label

In [4]:
def get_image_paths_and_labels(images_root):
    oil_dir = os.path.join(images_root, "Oil")
    no_oil_dir = os.path.join(images_root, "No_Oil")

    oil_files = [
        os.path.join(oil_dir, f)
        for f in os.listdir(oil_dir)
        if f.lower().endswith(".tif")
    ]

    no_oil_files = [
        os.path.join(no_oil_dir, f)
        for f in os.listdir(no_oil_dir)
        if f.lower().endswith(".tif")
    ]

    paths  = oil_files + no_oil_files
    labels = [1]*len(oil_files) + [0]*len(no_oil_files)

    return np.array(paths), np.array(labels)

In [5]:
def make_dataset(paths, labels, shuffle=False, augment_data=False):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    if shuffle:
        ds = ds.shuffle(len(paths), reshuffle_each_iteration=True)

    ds = ds.map(
        lambda x, y: (load_sar_tiff_tf(x), y)
    )

    if augment_data:
        ds = ds.map(augment)

    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(tf.data.AUTOTUNE)

    return ds

In [8]:
train_paths, train_labels = get_image_paths_and_labels(TRAIN_DIR)
test_paths,  test_labels  = get_image_paths_and_labels(TEST_DIR)

print("Train samples:", len(train_paths))
print("Test samples:", len(test_paths))

minority_mask = train_labels == 0
majority_mask = train_labels == 1

minority_paths  = train_paths[minority_mask]
minority_labels = train_labels[minority_mask]

majority_paths  = train_paths[majority_mask]
majority_labels = train_labels[majority_mask]

print("Minority (No_Oil):", len(minority_paths))
print("Majority (Oil):", len(majority_paths))

minority_ds = make_dataset(
    minority_paths,
    minority_labels,
    shuffle=True,
    augment_data=True
)

majority_ds = make_dataset(
    majority_paths,
    majority_labels,
    shuffle=True,
    augment_data=False
)

train_ds = minority_ds.concatenate(majority_ds)
train_ds = train_ds.prefetch(tf.data.AUTOTUNE)

test_ds = make_dataset(
    test_paths,
    test_labels,
    shuffle=False,
    augment_data=False
)

Train samples: 1885
Test samples: 300
Minority (No_Oil): 685
Majority (Oil): 1200


In [9]:
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1]),
    y=train_labels
)

class_weights = {
    0: class_weights[0],   # No_Oil (higher)
    1: class_weights[1],   # Oil (lower)
}

print("Class weights:", class_weights)

Class weights: {0: np.float64(1.3759124087591241), 1: np.float64(0.7854166666666667)}


In [10]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

def build_best_detection_cnn():
    model = Sequential()

    # 6 convolutional blocks
    #filters_list = [16, 32, 32, 64, 64, 64]
    filters_list = [16, 32, 64, 64, 64, 64]

    for i, filters in enumerate(filters_list):
        if i == 0:
            model.add(Conv2D(
                filters=filters,
                kernel_size=(3, 3),
                activation="relu",
                padding="same",
                input_shape=(512, 512, 2)
            ))
        else:
            model.add(Conv2D(
                filters=filters,
                kernel_size=(3, 3),
                activation="relu",
                padding="same"
            ))

        model.add(MaxPooling2D(pool_size=(2, 2)))

    model.add(Flatten())

    # Dense layer
    model.add(Dense(64, activation="relu"))

    # Dropout
    model.add(Dropout(0.4))

    # Output layer
    model.add(Dense(1, activation="sigmoid"))

    # Compile
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model


In [11]:
model = build_best_detection_cnn()
model.summary()

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 512, 512, 16)   │           304 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 256, 256, 16)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 256, 256, 32)   │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 128, 128, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 128, 128, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 64, 64, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 64, 64, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 32, 32, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 16, 16, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 8, 8, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 4096)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │       262,208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 396,497 (1.51 MB)

 Trainable params: 396,497 (1.51 MB)

 Non-trainable params: 0 (0.00 B)

In [12]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

early_stop = EarlyStopping(
    monitor='val_accuracy',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_accuracy',
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

In [13]:
history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=15,
    callbacks=[early_stop, reduce_lr],
    class_weight=class_weights,
)

Epoch 1/15
236/236 ━━━━━━━━━━━━━━━━━━━━ 2439s 10s/step - accuracy: 0.9610 - loss: 5.8347 - val_accuracy: 0.5000 - val_loss: 38.6977 - learning_rate: 0.0010
Epoch 2/15
236/236 ━━━━━━━━━━━━━━━━━━━━ 2440s 10s/step - accuracy: 0.7816 - loss: 9.4230 - val_accuracy: 0.5000 - val_loss: 24.7174 - learning_rate: 0.0010
Epoch 3/15
236/236 ━━━━━━━━━━━━━━━━━━━━ 2418s 10s/step - accuracy: 0.7555 - loss: 5.9655 - val_accuracy: 0.5000 - val_loss: 13.9447 - learning_rate: 0.0010
Epoch 4/15
236/236 ━━━━━━━━━━━━━━━━━━━━ 0s 9s/step - accuracy: 0.2711 - loss: 6.3380
Epoch 4: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
236/236 ━━━━━━━━━━━━━━━━━━━━ 2469s 10s/step - accuracy: 0.2724 - loss: 6.3256 - val_accuracy: 0.5000 - val_loss: 20.3965 - learning_rate: 0.0010
Epoch 5/15
236/236 ━━━━━━━━━━━━━━━━━━━━ 2377s 10s/step - accuracy: 0.3050 - loss: 6.5672 - val_accuracy: 0.5000 - val_loss: 2.6784 - learning_rate: 5.0000e-04
Epoch 6/15
236/236 ━━━━━━━━━━━━━━━━━━━━ 2412s 10s/step - accuracy: 

In [15]:
train_loss, train_acc = model.evaluate(train_ds)
print(f"Train Acc -> {train_acc}, Train Loss -> {train_loss}")

236/236 ━━━━━━━━━━━━━━━━━━━━ 2198s 9s/step - accuracy: 0.2712 - loss: 50.3090
Train Acc -> 0.6366047859191895, Train Loss -> 25.087867736816406


In [16]:
test_loss, test_acc = model.evaluate(test_ds)
print(f"Test Acc -> {test_acc}, Test Loss -> {test_loss}")

38/38 ━━━━━━━━━━━━━━━━━━━━ 277s 7s/step - accuracy: 0.8268 - loss: 13.5222
Test Acc -> 0.5, Test Loss -> 38.69770431518555


In [ ]:
import matplotlib.pyplot as plt

epochs = range(1, len(history.history["accuracy"]) + 1)

plt.figure(figsize=(10, 6))
plt.plot(epochs, history.history["accuracy"], label="Training Accuracy", linewidth=2)
plt.plot(epochs, history.history["val_accuracy"], label="Validation Accuracy", linewidth=2)

plt.title("Training vs Validation Accuracy", fontsize=16)
plt.xlabel("Epochs", fontsize=12)
plt.ylabel("Accuracy", fontsize=12)

plt.legend(fontsize=11)
plt.grid(True, linestyle="--", alpha=0.6)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(epochs, history.history["loss"], label="Training Loss", linewidth=2)
plt.plot(epochs, history.history["val_loss"], label="Validation Loss", linewidth=2)

plt.title("Training vs Validation Loss", fontsize=16)
plt.xlabel("Epochs", fontsize=12)
plt.ylabel("Loss", fontsize=12)

plt.legend(fontsize=11)
plt.grid(True, linestyle="--", alpha=0.6)

plt.tight_layout()
plt.show()